### Importing Libraries
This section imports all necessary libraries for data processing, analysis, and model building.

# Step 1: Import Necessary Libraries
This step imports all the required libraries for:
- Data manipulation and analysis: `pandas`, `numpy`
- Data visualization: `matplotlib`, `seaborn`
- Machine learning preprocessing and model evaluation: `scikit-learn`
- Deep learning model building: `tensorflow.keras`

These libraries provide the foundation for the project's implementation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

### Loading and Preprocessing Data
In this section, the dataset is loaded, irrelevant columns are dropped, and data is split into numeric and categorical types for preprocessing.

# Step 2: Load and Preprocess the Dataset
### Key Actions:
1. Load the dataset from a CSV file.
2. Drop irrelevant or redundant columns to reduce noise in the data.
3. Separate features (`X`) and target variable (`y`).
4. Identify numeric and categorical columns for appropriate preprocessing steps:
   - Numeric columns: Impute missing values with the mean and standardize.
   - Categorical columns: Impute missing values with the most frequent value and encode.

### Preprocessing Overview:
Using a `Pipeline` for transformations ensures the process is efficient and reproducible.

In [ ]:
data = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")

In [ ]:
missing_values = data.isnull().sum().sort_values(ascending=False)
missing_values = missing_values[missing_values > 0]
missing_values

In [ ]:
columns_to_drop = ["PoolQC", "MiscFeature", "Alley", "Fence", "MasVnrType", "FireplaceQu", "LotFrontage", "Id"]
data.drop(columns=columns_to_drop, axis=1, inplace=True)

X = data.drop("SalePrice", axis=1)
y = data["SalePrice"]


num = X.select_dtypes(include=["int64", "float64"]).columns
cat = X.select_dtypes(include=["object"]).columns


num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])
cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, num),
        ("cat", cat_transformer, cat)
    ]
)

X_preprocessed = preprocessor.fit_transform(X)

# Step 3: Split Data into Training and Testing Sets
The dataset is split into training (75%) and testing (25%) sets using the `train_test_split` method. This ensures:
- The model is trained on one portion of the data.
- The remaining data is used to evaluate its performance objectively.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_preprocessed, y, test_size=0.25, random_state=42)

### Building the Neural Network
A multi-layer neural network is defined, including layers for regularization and dropout to prevent overfitting.

# Step 4: Build the Neural Network
A deep learning model is designed using the `Sequential` API. The architecture includes:
- Dense layers with ReLU activation for feature extraction.
- Dropout layers to reduce overfitting.
- L2 regularization to constrain weights and enhance generalization.

The final layer outputs a single value for regression tasks.

In [ ]:
model = Sequential([
    Dense(256, activation='relu', input_dim=X_train.shape[1], kernel_regularizer=l2(0.01)),
    Dropout(0.3),
    Dense(128, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.3),
    Dense(64, activation='relu', kernel_regularizer=l2(0.01)),
    Dense(32, activation='relu', kernel_regularizer=l2(0.01)),
    Dense(1)
])


MyOptimizer = tf.keras.optimizers.RMSprop(learning_rate=0.001)
model.compile(optimizer=MyOptimizer, loss='mse', metrics=['mae'])

early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=100, batch_size=8, callbacks=[early_stop])

y_pred = model.predict(X_test)


# Step 5: Evaluate the Model
The trained model is evaluated on the test set to compute:
- Mean Squared Error (MSE): Measures the average squared difference between actual and predicted values.
- Mean Absolute Error (MAE): Represents the average absolute difference.

These metrics provide insights into the model's performance.

In [ ]:
test_loss, test_mae = model.evaluate(X_test, y_test)
print(f"Test Loss (MSE): {test_loss}")
print(f"Test MAE: {test_mae}")

In [ ]:
# Plot loss over epochs
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.legend()
plt.title("Loss Over Epochs")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.show()

In [ ]:
# Plot MAE over epochs
plt.plot(history.history["mae"], label="Training MAE")
plt.plot(history.history["val_mae"], label="Validation MAE")
plt.legend()
plt.title("Mean Absolute Error Over Epochs")
plt.xlabel("Epochs")
plt.ylabel("MAE")
plt.show()

#Step 6: Make Predictions and Prepare Submission File
This step demonstrates the process of generating predictions on the test dataset and preparing a submission file in the required format. It consists of the following key actions:

1. Load the Test Dataset
The test dataset is loaded using pandas.read_csv, and unnecessary columns (the same as those dropped during training) are removed to ensure consistency in the data format.

2. Preprocess the Test Data
The test data is transformed using the preprocessor object created earlier. This ensures that the same data preprocessing steps (e.g., scaling, encoding) applied to the training data are also applied to the test data.

3. Generate Predictions
The trained neural network model is used to make predictions on the preprocessed test data. The predictions are flattened to align with the expected submission format.

4. Prepare the Submission File
The predictions are added to the sample_submission dataframe under the column SalePrice. This dataframe is then saved as a CSV file named submission.csv. The index=False parameter ensures the index column is not included in the final output.

In [ ]:
test_data = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")
test_data.drop(columns=columns_to_drop[:-1], axis=1, inplace=True)


test_preprocessed = preprocessor.transform(test_data)


predicted_labels = model.predict(test_preprocessed).flatten()

sample_submission = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv")
sample_submission["SalePrice"] = predicted_labels
sample_submission.to_csv("submission.csv", index=False)
print("The submission file has been created.")

In [ ]:
sample_submission